# COMP3122 — Lab 06 — Your First Model
### Week 6 · Chapter 6: scikit-learn and Linear Regression

**Applied Data Science**

**This lab is graded.** It covers Chapter 6 in the order the chapter covers it.

**How it works:** each task has a cell for your answer, followed by a cell that checks it. Run the check cell and it prints `PASS`, or shows you what it expected.

**Derive your answers; do not type them.** Most of what this lab asks for is a number, and a number typed in from the chapter passes the same check as a number you computed. Several tasks are built to catch that anyway — Task 3.3 asks you to reproduce a prediction by hand and checks it against `.predict()`, and Task 7.1 asks which feature is more useful and settles it by removing each one. But mostly this is on you, and the person you would be cheating is the one sitting the quiz.

**Four tasks ask you to write down a prediction before you run anything** — Tasks 2.4, 4.2, 4.3 and 7.1. Do that. Being wrong on purpose, once, is worth more than being right by accident four times, and one of those four surprises most people who have read the chapter carefully.

**Part 4 is the one to slow down for.** The train/test split is the most important rule in the chapter, and Task 4.3 is where it stops being a slogan.

**Part 6 needs the internet, once.** The setup cell downloads about 400 KB of real housing data and caches it. If it cannot, the Part 6 checks will say `SKIPPED` and tell you why — nothing else in the lab is affected, so carry on and come back to Part 6 when you are connected.

Work in order. The setup cell builds `sizes`, `prices`, `houses`, `homes` and `workers` and defines the checkers, so run it first. **Do not reassign those names** — build your answers into the variables each task asks for. If a check will not pass after five minutes, ask.

---
## Setup — run this first

In [ ]:
# Run this cell once, before you start. It builds the data and defines the checkers.
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# ---- the six houses from Chapter 6 §6.2 ------------------------------------
sizes  = np.array([50, 70, 90, 110, 130, 150])     # floor size in square metres
prices = np.array([150, 200, 250, 280, 340, 370])  # price in thousands of dollars
house_names = ["A", "B", "C", "D", "E", "F"]

houses = pd.DataFrame({"house": house_names, "size_m2": sizes, "price_k": prices})

# ---- 200 homes, so that a train/test split has something to measure --------
# Six rows are enough to check arithmetic by hand and far too few to score a
# model on. Parts 4 and 5 use this larger table instead. It is generated from a
# fixed seed, so every student gets exactly the same 200 rows.
_rng   = np.random.default_rng(3122)
_area  = _rng.integers(40, 200, 200)
_price = 40 + 2.2 * _area + _rng.normal(0, 25, 200)
homes  = pd.DataFrame({"area_m2": _area, "price_k": _price.round(1)})

# ---- five workers, for the shape practice in Part 1 ------------------------
workers = pd.DataFrame({
    "years":  [1, 3, 5, 7, 9],
    "salary": [42, 50, 61, 70, 78],     # thousands
})

# ---- California Housing, for Part 6 ----------------------------------------
# Part 6 is the only part that needs this. Everything else runs offline. The
# first call downloads about 400 KB and caches it; later calls read the cache.
# If it cannot load, Parts 1-5 and 7 are unaffected and the Part 6 checks will
# tell you why rather than showing you a traceback.
# Untouched copies, so the Task 7.3 guard can tell whether anything was reassigned.
SIZES_AT_START, PRICES_AT_START = sizes.copy(), prices.copy()
HOUSES_AT_START, HOMES_AT_START = houses.copy(), homes.copy()

# The rows a correct Task 4.1 split puts in the test set, from the untouched table.
TEST_ROWS_41 = sorted(train_test_split(HOMES_AT_START, test_size=0.2,
                                       random_state=42)[1].index)

housing_ok = False
try:
    from sklearn.datasets import fetch_california_housing
    _cal = fetch_california_housing(as_frame=True)
    cal_X, cal_y = _cal.data, _cal.target
    housing_ok = True
except Exception as _e:
    cal_X = cal_y = None
    _cal_err = f"{type(_e).__name__}: {_e}"


def check(label, got, want):
    """Compare your answer with the expected value. Prints PASS or NOT YET.

    `got` may be a value, or a zero-argument function producing one. The second
    form is used wherever the check has to reach into your answer first — read an
    attribute, round a number, index into an array. Before the answer is filled in
    that would raise, and a traceback where a hint should be is not helpful, so it
    is caught and reported as NOT YET instead.
    """
    if callable(got):
        try:
            got = got()
        except Exception as e:
            print(f"NOT YET  {label}")
            print(f"        your answer raised {type(e).__name__}: {e}")
            return

    def _fail(why=None):
        print(f"NOT YET  {label}")
        if why:
            print(f"        {why}")
        print(f"        you gave: {got!r}")
        print(f"        expected: {want!r}")

    PANDAS = (pd.Series, pd.DataFrame)
    if isinstance(want, PANDAS) or isinstance(got, PANDAS):
        if type(got) is not type(want):
            return _fail(f"wrong type — got {type(got).__name__}, expected {type(want).__name__}")
        if list(got.index) != list(want.index):
            return _fail("the row labels do not match")
        if isinstance(want, pd.DataFrame) and list(got.columns) != list(want.columns):
            return _fail("the column labels do not match")
        try:
            ok = bool(got.equals(want))
        except Exception:
            ok = False
    else:
        g, w = np.asarray(got), np.asarray(want)
        try:
            if g.shape == w.shape and g.dtype.kind in "fc" and w.dtype.kind in "fc":
                ok = bool(np.allclose(g, w))   # decimals: allow a tiny rounding difference
            else:
                ok = bool(np.array_equal(g, w))
        except Exception:
            ok = False

    if ok:
        print(f"PASS  {label}")
    else:
        _fail()


def check_housing(label):
    """Report the Part 6 tasks honestly when the dataset could not be loaded.

    Returns True when it is safe to check a Part 6 answer. Without this, a
    student with no internet gets a wall of NameErrors in Part 6 and no idea
    which of them is the real problem.
    """
    if housing_ok:
        return True
    print(f"SKIPPED  {label}")
    print(f"        California Housing did not load in the setup cell ({_cal_err})")
    print(f"        Parts 1-5 and 7 do not need it. Run the setup cell once on a")
    print(f"        machine with internet and the cached copy stays on that machine.")
    return False


def same_rows(X_train, X_test, y_train, y_test):
    """True when every row of the four pieces still holds its own area and price.

    Task 4.1 uses it: each piece is looked up in the untouched 200 homes by row
    label, and the numbers are compared as numbers, so 120 and 120.0 count as equal.
    """
    ref = HOMES_AT_START

    def same(got, col):
        want = ref.loc[got.index, col].to_numpy(dtype=float)
        return np.allclose(np.asarray(got, dtype=float), want)

    return bool(same(X_train["area_m2"], "area_m2") and same(X_test["area_m2"], "area_m2")
                and same(y_train, "price_k") and same(y_test, "price_k"))


def error_message(fn):
    """Run fn() and return the whole error message it raised, or 'no error'.

    Task 1.2 uses it to show the full message without stopping the notebook.
    """
    try:
        fn()
    except Exception as e:
        return f"{type(e).__name__}: {e}"
    return "no error"


def raises(fn):
    """Run fn() and return the name of the exception it raised, or 'no error'.

    Several tasks ask you to find out what a wrong line actually does. Guessing
    is not the same as knowing, and this is how you check.
    """
    try:
        fn()
    except Exception as e:
        return type(e).__name__
    return "no error"


print("Ready. houses:", houses.shape, " homes:", homes.shape,
      " workers:", workers.shape)
print("California Housing:", (f"loaded {cal_X.shape}" if housing_ok
                              else "NOT loaded — Part 6 will say so"))
import sklearn
print("scikit-learn", sklearn.__version__, " pandas", pd.__version__,
      " numpy", np.__version__)


---
## Part 1 — Features, target, and shape  ·  §6.2

Capital `X` is the features. Lowercase `y` is the target. That is not decoration — it is
a shape rule, and breaking it is the first error almost everybody hits.

**Task 1.1.** Turn the six houses into the two things scikit-learn wants.

Build `X11` from `sizes` as a **2-dimensional** feature table with one column, and `y11`
from `prices` as a **1-dimensional** target.

Use `.reshape(-1, 1)` for the feature. Do not reshape the target.

In [ ]:
X11 = None
y11 = None

In [ ]:
check('1.1 X is 2D',        lambda: X11.ndim, 2)
check('1.1 X shape',        lambda: X11.shape, (6, 1))
check('1.1 y is 1D',        lambda: y11.ndim, 1)
check('1.1 y shape',        lambda: y11.shape, (6,))
check('1.1 X values kept',  lambda: X11.ravel().tolist(), [50, 70, 90, 110, 130, 150])
check('1.1 y is the prices', lambda: list(y11), [150, 200, 250, 280, 340, 370])

**Task 1.2.** Find out what the shape mistake actually does, rather than guessing.

Use the `raises` helper to capture the **name of the exception** you get from fitting a model
on the flat `sizes` array:

```python
t12_error = raises(lambda: LinearRegression().fit(sizes, prices))
```

`raises` keeps only the exception's name. To read the **whole message**, the answer cell
for this task starts with a line that prints it, using the `error_message` helper from the
setup cell.
Run it and read every line of the output.

Then answer `t12_message_helps`: does scikit-learn's error message tell you the fix?
`True` or `False`.

In [ ]:
# explain: prints the whole error message, without stopping the notebook
print(error_message(lambda: LinearRegression().fit(sizes, prices)))

t12_error = None
t12_message_helps = None

In [ ]:
check('1.2 exception name',   t12_error, 'ValueError')
check('1.2 message helps?',   t12_message_helps, True)

**Task 1.3.** The DataFrame spelling of the same rule.

From the `houses` table, take the `size_m2` column two ways and record what you get:

- `t13_single_shape` — the shape of `houses["size_m2"]`
- `t13_double_shape` — the shape of `houses[["size_m2"]]`
- `t13_usable` — which of the two scikit-learn accepts as `X`: the string `'single'`
  or `'double'`

In [ ]:
t13_single_shape = None
t13_double_shape = None
t13_usable = None

In [ ]:
check('1.3 single brackets', t13_single_shape, (6,))
check('1.3 double brackets', t13_double_shape, (6, 1))
check('1.3 which is usable', t13_usable, 'double')

**Task 1.4.** Same move, different data — the five workers.

Build `X14` (2D) from the `years` column of `workers`, and `y14` (1D) from `salary`.
Use the DataFrame form, not `reshape`.

In [ ]:
X14 = None
y14 = None

In [ ]:
check('1.4 X14 shape',  lambda: X14.shape, (5, 1))
check('1.4 y14 shape',  lambda: y14.shape, (5,))
check('1.4 X14 is a DataFrame', lambda: type(X14).__name__, 'DataFrame')
check('1.4 y14 is a Series',    lambda: type(y14).__name__, 'Series')
check('1.4 X14 is the years column', lambda: list(X14.columns), ['years'])
check('1.4 y14 is the salary column', lambda: y14.name, 'salary')

---
## Part 2 — Residuals and least squares  ·  §6.3

A residual is one prediction's error. Square them, add them up, and you have a single
number that scores a whole line. Least squares picks the line where that number is
smallest — and this part is where you see that it really is a search over lines.

**Task 2.1.** Score a line the chapter picked by eye.

The candidate is `price = 50 + 2 × size`. Using `sizes`, compute:

- `t21_pred` — its prediction for each of the six houses (an array)
- `t21_resid` — the residual for each house, **actual minus predicted**

Compute both. Do not type the numbers in.

In [ ]:
t21_pred = None
t21_resid = None

In [ ]:
check('2.1 predictions', lambda: t21_pred.tolist(), [150, 190, 230, 270, 310, 350])
check('2.1 residuals',   lambda: t21_resid.tolist(), [0, 10, 20, 10, 30, 20])

**Task 2.2.** Turn the residuals into one number.

Compute `t22_ssr` — the sum of the **squared** residuals over all six houses.

Derive it from `t21_resid`. The chapter's hand table sums only three houses and gets 1300;
you are summing all six, so you should not get 1300.

In [ ]:
t22_ssr = None

In [ ]:
check('2.2 SSR over all six', t22_ssr, 1900)

**Task 2.3.** Score a second line, and let the numbers choose.

Try `price = 40 + 2.2 × size`. Compute its SSR over all six houses into `t23_ssr`
(round to 2 decimals), then set `t23_better` to the string `'first'` or `'second'` —
whichever line least squares prefers.

In [ ]:
t23_ssr = None
t23_better = None

In [ ]:
check('2.3 second line SSR', t23_ssr, 380.0)
check('2.3 which line wins', t23_better, 'second')

**Task 2.4.** Predict first, then measure.

Before you run anything, write down what you expect. Then compute, from `t21_resid`:

- `t24_positive` — how many residuals are greater than zero
- `t24_zero` — how many are exactly zero
- `t24_negative` — how many are less than zero

Then set `t24_where` to `'above'`, `'below'` or `'through'`: where does this line sit
relative to the data?

In [ ]:
t24_positive = None
t24_zero = None
t24_negative = None
t24_where = None

In [ ]:
check('2.4 positive residuals', t24_positive, 5)
check('2.4 zero residuals',     t24_zero, 1)
check('2.4 negative residuals', t24_negative, 0)
check('2.4 where the line sits', t24_where, 'below')

---
## Part 3 — fit, predict, score  ·  §6.4

Three methods. Learn them here and you can drive almost every supervised model in the
library — including next week's.

**Task 3.1.** Fit a model and read what it learned.

Create a `LinearRegression`, fit it on `X11` and `y11` from Task 1.1, and store it as
`m31`. Then read out, rounded to 4 decimals:

- `t31_coef` — the coefficient
- `t31_intercept` — the intercept

`coef_` is an array even with one feature, so index into it.

In [ ]:
m31 = None
t31_coef = None
t31_intercept = None

In [ ]:
check('3.1 coefficient', t31_coef, 2.2143)
check('3.1 intercept',   t31_intercept, 43.5714)
check('3.1 SSR beats both Part 2 lines',
      lambda: round(float(((y11 - m31.predict(X11)) ** 2).sum()), 2), 228.57)

**Task 3.2.** Predict two houses nobody measured.

Use `m31` to predict the price of a 100 m² house and a 125 m² house. Store the two
predictions, rounded to 4 decimals, in the list `t32_pred`.

Remember the shape rule — `predict` wants a 2D array too.

In [ ]:
t32_pred = None

In [ ]:
check('3.2 two predictions', t32_pred, [265.0, 320.3571])
check('3.2 predicted from the fitted model, not typed',
      lambda: [round(float(v), 4) for v in m31.predict(np.array([[100], [125]]))],
      t32_pred)

**Task 3.3.** Check the model's arithmetic by hand.

A linear model is `prediction = intercept + coefficient × feature`, and nothing else.
Compute the 100 m² prediction yourself from `m31.coef_` and `m31.intercept_` into
`t33_by_hand` (rounded to 4 decimals), and set `t33_matches` to `True` or `False`:
does it match what `.predict()` returned?

In [ ]:
t33_by_hand = None
t33_matches = None

In [ ]:
check('3.3 by hand',          t33_by_hand, 265.0)
check('3.3 matches predict()', t33_matches, True)
check('3.3 really identical',
      lambda: round(float(m31.intercept_ + m31.coef_[0] * 100), 4)
              == round(float(m31.predict([[100]])[0]), 4), True)

**Task 3.4.** The trailing underscore.

scikit-learn puts everything a model **learned** into attributes ending in `_`. Before
fitting, they do not exist.

Create a fresh, unfitted `LinearRegression()` as `m34`, then record:

- `t34_before` — `hasattr(m34, "coef_")` before fitting
- `t34_error` — the exception name from `raises(lambda: m34.coef_)`
- `t34_after` — `hasattr(...)` after you fit it on `X11`, `y11`

In [ ]:
m34 = None
t34_before = None
t34_error = None
t34_after = None

In [ ]:
check('3.4 before fitting', t34_before, False)
check('3.4 exception name', t34_error, 'AttributeError')
check('3.4 after fitting',  t34_after, True)

---
## Part 4 — The train/test split  ·  §6.5

**Slow down here.** This is the most important rule in the chapter, and Task 4.3 is the
one most likely to surprise you. Six houses are too few to split meaningfully, so this
part uses `homes` — 200 rows, built in the setup cell.

**Task 4.1.** Split, and look at what comes back.

Split `homes[["area_m2"]]` and `homes["price_k"]` with `test_size=0.2` and
`random_state=42`. Unpack into `X41_train, X41_test, y41_train, y41_test` — **in that
order**, which is the order `train_test_split` returns.

Then record the four row counts, in that same order, in the list `t41_rows`.

In [ ]:
X41_train = X41_test = y41_train = y41_test = None
t41_rows = None

In [ ]:
check('4.1 four row counts', t41_rows, [160, 40, 160, 40])
check('4.1 features stayed 2D',
      lambda: (X41_train.ndim, X41_test.ndim), (2, 2))
check('4.1 targets stayed 1D',
      lambda: (y41_train.ndim, y41_test.ndim), (1, 1))
check('4.1 no row is in both train and test',
      lambda: len(set(X41_train.index) & set(X41_test.index)), 0)
check('4.1 every one of the 200 rows is used exactly once',
      lambda: sorted(list(X41_train.index) + list(X41_test.index)),
      list(HOMES_AT_START.index))
check('4.1 the test rows are the ones random_state=42 picks',
      lambda: sorted(X41_test.index), TEST_ROWS_41)
check('4.1 both feature tables hold only the area; both targets are the price',
      lambda: ', '.join(X41_train.columns) + ' | ' + ', '.join(X41_test.columns)
              + ' -> ' + str(y41_train.name) + ' | ' + str(y41_test.name),
      'area_m2 | area_m2 -> price_k | price_k')
check('4.1 features and targets still line up row for row',
      lambda: bool(X41_train.index.equals(y41_train.index)
                   and X41_test.index.equals(y41_test.index)), True)
check('4.1 each row keeps its own area and price',
      lambda: same_rows(X41_train, X41_test, y41_train, y41_test), True)

**Task 4.2.** Predict first, then measure — the rounding.

Write down your answer before running anything: split the **six houses** (`X11`, `y11`)
with `test_size=0.2`. How many test houses do you get?

20% of 6 is 1.2. Now measure it into `t42_test_rows`, and set `t42_rule` to `'up'` or
`'down'` — which way does scikit-learn round the test count?

In [ ]:
t42_test_rows = None
t42_rule = None

In [ ]:
check('4.2 test rows from six', t42_test_rows, 2)
check('4.2 rounding direction', t42_rule, 'up')

**Task 4.3.** Predict first, then measure — the one that matters.

Fit a fresh model on the **training** rows from Task 4.1 only, and store it as `m43`.
Then compute both scores, rounded to 4 decimals:

- `t43_train_score` — `m43.score()` on the training rows
- `t43_test_score` — `m43.score()` on the test rows

**Before you run it, write down which you expect to be higher.** Then record what actually
happened in `t43_higher`: the string `'train'` or `'test'`.

In [ ]:
m43 = None
t43_train_score = None
t43_test_score = None
t43_higher = None

In [ ]:
check('4.3 training score', t43_train_score, 0.9364)
check('4.3 test score',     t43_test_score, 0.9566)
check('4.3 which was higher', t43_higher, 'test')

**Task 4.4.** What `random_state` buys you.

Split `homes` twice more, both with `test_size=0.2`: once with `random_state=7`, once with
`random_state=7` again, and once with `random_state=8`. Record:

- `t44_same_seed_identical` — do the two seed-7 test sets contain the same rows? `True`/`False`
- `t44_diff_seed_identical` — does seed 8 give the same test rows as seed 7? `True`/`False`

In [ ]:
t44_same_seed_identical = None
t44_diff_seed_identical = None

In [ ]:
check('4.4 same seed, same rows',      t44_same_seed_identical, True)
check('4.4 different seed, same rows?', t44_diff_seed_identical, False)

---
## Part 5 — R² and MSE  ·  §6.6

Two numbers, two different questions. MSE asks how big the errors are. R² asks how much
better than guessing the average you did.

**Task 5.1.** Build R² from parts.

Using `m31`'s predictions on all six houses, compute, each rounded to 4 decimals:

- `t51_ssr` — the sum of squared residuals of the fitted line
- `t51_sst` — the sum of squared residuals of always guessing the **mean** of `y11`
- `t51_r2` — `1 - SSR/SST`, computed from your two numbers

Do not use `r2_score` for `t51_r2` — the check compares your hand calculation against it.

In [ ]:
t51_ssr = None
t51_sst = None
t51_r2 = None

In [ ]:
check('5.1 SSR', t51_ssr, 228.5714)
check('5.1 SST', t51_sst, 34550.0)
check('5.1 R-squared by hand', t51_r2, 0.9934)
check('5.1 matches r2_score',
      lambda: round(r2_score(y11, m31.predict(X11)), 4), t51_r2)
check('5.1 matches .score()',
      lambda: round(m31.score(X11, y11), 4), t51_r2)

**Task 5.2.** Is the mean really the best baseline?

R² measures your line against always guessing the average. Test that the average is the best
constant to guess: for every whole number from 0 to 599, compute the total squared error of
guessing it for all six houses, and store the **winning constant** in `t52_best`.

Then set `t52_is_mean` to `True` or `False`: is it the mean of `prices`?

In [ ]:
t52_best = None
t52_is_mean = None

In [ ]:
check('5.2 best constant', t52_best, 265)
check('5.2 it is the mean', t52_is_mean, True)
check('5.2 and its cost is SST',
      lambda: round(float(((prices - t52_best) ** 2).sum()), 4), 34550.0)

**Task 5.3.** The argument order, and why one metric hides it.

Both `r2_score` and `mean_squared_error` take the **real values first**. Using the Task 2.1
candidate predictions `t21_pred`, compute all four, rounded to 4 decimals:

- `t53_r2_right`, `t53_r2_wrong`
- `t53_mse_right`, `t53_mse_wrong`

Then set `t53_which_hides` to `'r2_score'` or `'mean_squared_error'` — which one gives you
**no clue** that you got the order wrong?

In [ ]:
t53_r2_right = None
t53_r2_wrong = None
t53_mse_right = None
t53_mse_wrong = None
t53_which_hides = None

In [ ]:
check('5.3 r2 correct order',   t53_r2_right, 0.945)
check('5.3 r2 swapped',         t53_r2_wrong, 0.9321)
check('5.3 mse correct order',  t53_mse_right, 316.6667)
check('5.3 mse swapped',        t53_mse_wrong, 316.6667)
check('5.3 which one hides it', t53_which_hides, 'mean_squared_error')
check('5.3 so r2 changed and mse did not',
      lambda: (t53_r2_right != t53_r2_wrong, t53_mse_right == t53_mse_wrong), (True, True))

**Task 5.4.** Below zero.

R² goes negative when a model does **worse** than guessing the average. Make that happen:
predict a flat 500 for all six houses, store the R² in `t54_r2` (rounded to 4 decimals),
and set `t54_meaning` to `'worse than the mean'` or `'the model is broken'`.

In [ ]:
t54_r2 = None
t54_meaning = None

In [ ]:
check('5.4 negative R-squared', t54_r2, -9.5904)
check('5.4 it is below zero',   lambda: t54_r2 < 0, True)
check('5.4 what it means',      t54_meaning, 'worse than the mean')

---
## Part 6 — The whole workflow on real data  ·  §6.7

**This is the only part that needs the internet**, and only the first time — the setup
cell downloads about 400 KB and caches it. If it did not load, the checks below will say
so and the rest of the lab is unaffected.

The data is `cal_X` and `cal_y`, built in the setup cell.

**Task 6.1.** Look before you model.

Record the shape of the problem:

- `t61_rows` — how many block groups
- `t61_features` — how many feature columns
- `t61_first_feature` — the name of the first column, as a string

In [ ]:
t61_rows = None
t61_features = None
t61_first_feature = None

In [ ]:
if check_housing('6.1'):
    check('6.1 rows',          t61_rows, 20640)
    check('6.1 features',      t61_features, 8)
    check('6.1 first feature', t61_first_feature, 'MedInc')

**Task 6.2.** Find the wall that `describe()` does not mention.

The target has a ceiling: a group of block groups whose median value was capped when the
census was published. Derive, do not type:

- `t62_ceiling` — the maximum value of `cal_y`
- `t62_capped` — how many rows sit at that maximum
- `t62_percent` — what percentage of all rows that is, rounded to 1 decimal

In [ ]:
t62_ceiling = None
t62_capped = None
t62_percent = None

In [ ]:
if check_housing('6.2'):
    check('6.2 the ceiling',     t62_ceiling, 5.00001)
    check('6.2 rows capped',     t62_capped, 965)
    check('6.2 as a percentage', t62_percent, 4.7)

**Task 6.3.** One feature, four steps.

Split, fit and score using **median income alone**:

1. take `MedInc` from `cal_X`, keeping it 2D
2. split with `test_size=0.2`, `random_state=42`
3. fit on the training rows only
4. score on the test rows, into `t63_score` (rounded to 4 decimals)

Also record `t63_coef_sign` as `'positive'` or `'negative'`.

In [ ]:
t63_score = None
t63_coef_sign = None

In [ ]:
if check_housing('6.3'):
    check('6.3 test R-squared', t63_score, 0.4589)
    check('6.3 coefficient sign', t63_coef_sign, 'positive')

**Task 6.4.** Eight features, and the same four steps.

Repeat Task 6.3 with the **whole** of `cal_X`. Store the test R² in `t64_score`
(4 decimals), and pair the learned coefficients with the feature names in a Series
called `t64_coefs`.

Then set `t64_biggest` to the name of the feature with the largest coefficient.

In [ ]:
t64_score = None
t64_coefs = None
t64_biggest = None

In [ ]:
if check_housing('6.4'):
    check('6.4 test R-squared',   t64_score, 0.5758)
    check('6.4 eight coefficients', lambda: len(t64_coefs), 8)
    check('6.4 labelled by the eight feature names',
          lambda: sorted(t64_coefs.index), sorted(cal_X.columns))
    check('6.4 each coefficient sits next to its own feature',
          lambda: t64_coefs[list(cal_X.columns)].round(4).tolist(),
          [0.4487, 0.0097, -0.1233, 0.7831, -0.0, -0.0035, -0.4198, -0.4337])
    check('6.4 largest coefficient', t64_biggest, 'AveBedrms')
    check('6.4 eight features beat one', lambda: t64_score > t63_score, True)

**Task 6.5.** What the score hides.

R² of 0.58 is one number for 4,128 test block groups, and it is not equally true of all of
them. Using the eight-feature model from Task 6.4, compare its **mean absolute error** on
the capped test rows against the rest.

**Mean absolute error (MAE)** is the average distance between the true values and the
predictions: `np.abs(actual - predicted).mean()`. It does not square anything, so it stays
in the target's units. Here one unit is 100,000 dollars.

- `t65_mae_capped` — mean absolute error where the true value is at the ceiling
- `t65_mae_rest` — mean absolute error everywhere else

Both rounded to 4 decimals. Then set `t65_worse` to `'capped'` or `'rest'`.

In [ ]:
t65_mae_capped = None
t65_mae_rest = None
t65_worse = None

In [ ]:
if check_housing('6.5'):
    check('6.5 error on capped rows', t65_mae_capped, 1.4957)
    check('6.5 error on the rest',    t65_mae_rest, 0.4896)
    check('6.5 which is worse',       t65_worse, 'capped')
    check('6.5 and it is much worse',
          lambda: round(t65_mae_capped / t65_mae_rest, 1), 3.1)

---
## Part 7 — The traps  ·  §6.9

Four of the five traps in this chapter produce no error message. This part is the one
that reads most like sound reasoning and is not.

**Task 7.1.** Predict first, then measure — is the bigger coefficient the better feature?

Two features predict the same target. `income_usd` is measured in dollars; `bedrooms` is a
small count. They are built for you below.

Fit three models and record, each rounded to 4 decimals:

- `t71_coef_income`, `t71_coef_bedrooms` — the two coefficients from the **both-features** model
- `t71_r2_income_only`, `t71_r2_bedrooms_only` — the R² of each feature used **alone**

Then set `t71_more_useful` to `'income_usd'` or `'bedrooms'`.

```python
_r = np.random.default_rng(6)
income_usd = _r.normal(50_000, 12_000, 500)
bedrooms   = _r.integers(1, 6, 500).astype(float)
t71_target = 0.004 * income_usd + 2.0 * bedrooms + _r.normal(0, 5, 500)
```

In [ ]:
_r = np.random.default_rng(6)
income_usd = _r.normal(50_000, 12_000, 500)
bedrooms   = _r.integers(1, 6, 500).astype(float)
t71_target = 0.004 * income_usd + 2.0 * bedrooms + _r.normal(0, 5, 500)

t71_coef_income = None
t71_coef_bedrooms = None
t71_r2_income_only = None
t71_r2_bedrooms_only = None
t71_more_useful = None

In [ ]:
check('7.1 income coefficient',   t71_coef_income, 0.004)
check('7.1 bedrooms coefficient', t71_coef_bedrooms, 1.8303)
check('7.1 bedrooms coefficient is far larger',
      lambda: t71_coef_bedrooms > 100 * t71_coef_income, True)
check('7.1 R-squared, income alone',   t71_r2_income_only, 0.9867)
check('7.1 R-squared, bedrooms alone', t71_r2_bedrooms_only, 0.0102)
check('7.1 which feature is more useful', t71_more_useful, 'income_usd')

**Task 7.2.** Read four scenarios and say which number to trust.

For each, answer `'trustworthy'` or `'not trustworthy'` as an estimate of how the model will
perform on new data:

- `t72_a` — fitted on all 1,000 rows, scored on the same 1,000 rows, R² = 0.97
- `t72_b` — fitted on 800 rows, scored on 200 held-back rows, R² = 0.61
- `t72_c` — fitted on 800 rows, scored on 200 held-back rows, R² = 0.61, then the model was
  changed and re-scored on those same 200 rows twenty times until R² reached 0.74
- `t72_d` — fitted on 800 rows, scored on 200 held-back rows, R² = −0.12

In [ ]:
t72_a = None
t72_b = None
t72_c = None
t72_d = None

In [ ]:
check('7.2 a — scored on training rows', t72_a, 'not trustworthy')
check('7.2 b — scored once on held-back rows', t72_b, 'trustworthy')
check('7.2 c — test set reused twenty times',  t72_c, 'not trustworthy')
check('7.2 d — honest score of a bad model',   t72_d, 'trustworthy')

**Task 7.3 — GUARD.** Nothing to write.

The check below reads `sizes`, `prices`, `houses` and `homes` directly and confirms they
still hold what the setup cell put there. It passes on an untouched notebook, so it is not
evidence of work — it exists to fail loudly if something above overwrote the shared data.

In [ ]:
# Nothing to write for this task.

In [ ]:
check('7.3 GUARD sizes untouched',
      lambda: bool(np.array_equal(sizes, SIZES_AT_START)), True)
check('7.3 GUARD prices untouched',
      lambda: bool(np.array_equal(prices, PRICES_AT_START)), True)
check('7.3 GUARD houses untouched', lambda: houses.equals(HOUSES_AT_START), True)
check('7.3 GUARD homes untouched',  lambda: homes.equals(HOMES_AT_START), True)

---
## Finish

Choose **Kernel → Restart Kernel and Run All Cells** one last time. If every check prints `PASS` from a clean start, your work is reproducible and you are done. (Part 6's checks may print `SKIPPED` if you are offline — see the setup cell output.)

**If Task 4.3 surprised you, that is the intended difficulty.** The test score came out higher than the training score, and the chapter told you a training score is optimistic. Both are true. A training score is optimistic **on average**, not on every individual split — split the real housing data three hundred ways and the test score wins in nearly half of them. The reason you still cannot skip the split is that you have no way of telling, from a training score alone, which of the two cases you are in. Re-read §6.5 if that is not yet comfortable; it is the idea the rest of the book is built on.

**Two answers to carry into the quiz**, because both are asked in ways that reward understanding rather than recall:

- A coefficient's size is set by the **units** its feature is measured in. It is not a ranking of importance (Task 7.1).
- `mean_squared_error` gives the **same** number when you swap its arguments, so it cannot warn you that you got the order wrong. `r2_score` at least changes (Task 5.3).

Next week the target stops being a number and becomes a category, and almost all of this carries straight over.

---
## Your submission line

Run the cell below and paste **the line it prints** into *Lab 6 — submit* on D2L. Nothing else — not the address bar of this notebook, which carries a port number that stops working when your server restarts. The line is built from your account name, so it always works.

In [ ]:
# Run this cell. Copy the line it prints into 'Lab 6 -- submit' on D2L. Nothing else.
from pathlib import Path
account = Path.home().name                                   # your gblearn account name
print(f'http://{account}.gblearn.com/python/?open=comp3122/labs/lab06/COMP3122-Lab06-Student.ipynb')